# Phase 2: Feature Engineering

### Step 1: Load the Processed Data

In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, PolynomialFeatures

df = pd.read_csv('../data/processed/final_dengue_weather.csv')
df['Date'] = pd.to_datetime(df['Date'])
df = df.sort_values(['Region', 'Date']).reset_index(drop=True)

display(df.head())

,Month,Year,Region,Dengue_Cases,Dengue_Deaths,Date,rfh,rfh_avg
0,January,2016,BARMM,126,2,2016-01-01,21.523342,40.213978
1,February,2016,BARMM,92,2,2016-02-01,44.948402,38.875594
2,March,2016,BARMM,107,3,2016-03-01,35.624078,51.846384
3,April,2016,BARMM,109,4,2016-04-01,91.201476,53.762189
4,May,2016,BARMM,165,4,2016-05-01,241.466832,79.995030


### Step 2: Creating the Classification Target and Lag Features

In [2]:
thresholds = df.groupby('Region')['Dengue_Cases'].transform(lambda x: x.quantile(0.75))
df['Is_Outbreak'] = (df['Dengue_Cases'] > thresholds).astype(int)

df['rfh_lag1'] = df.groupby('Region')['rfh'].shift(1)
df['rfh_lag1'] = df.groupby('Region')['rfh_lag1'].transform(lambda x: x.bfill())

display(df[['Date', 'Region', 'Dengue_Cases', 'Is_Outbreak', 'rfh', 'rfh_lag1']].head())

,Date,Region,Dengue_Cases,Is_Outbreak,rfh,rfh_lag1
0,2016-01-01,BARMM,126,0,21.523342,21.523342
1,2016-02-01,BARMM,92,0,44.948402,21.523342
2,2016-03-01,BARMM,107,0,35.624078,44.948402
3,2016-04-01,BARMM,109,0,91.201476,35.624078
4,2016-05-01,BARMM,165,0,241.466832,91.201476


### Step 3: Polynomial Features

In [3]:
weather_cols = ['rfh', 'rfh_avg', 'rfh_lag1']

poly = PolynomialFeatures(degree=2, include_bias=False)
poly_features = poly.fit_transform(df[weather_cols])
poly_col_names = poly.get_feature_names_out(weather_cols)

poly_df = pd.DataFrame(poly_features, columns=poly_col_names)

df = pd.concat([df.drop(columns=weather_cols), poly_df], axis=1)

### Step 4: Feature Scaling and Export

In [4]:
scaler = StandardScaler()

df[poly_col_names] = scaler.fit_transform(df[poly_col_names])

df.to_csv('../data/processed/engineered_dengue_weather.csv', index=False)

display(df.head())
print("Phase 2 Complete! Engineered dataset shape:", df.shape)

,Month,Year,Region,Dengue_Cases,Dengue_Deaths,Date,Is_Outbreak,rfh,rfh_avg,rfh_lag1,rfh^2,rfh rfh_avg,rfh rfh_lag1,rfh_avg^2,rfh_avg rfh_lag1,rfh_lag1^2
0,January,2016,BARMM,126,2,2016-01-01,0,-1.292139,-0.889130,-1.266020,-0.722038,-0.870976,-0.797245,-0.806988,-0.931584,-0.709404
1,February,2016,BARMM,92,2,2016-02-01,0,-1.140636,-0.922607,-1.266020,-0.706514,-0.833938,-0.790757,-0.821920,-0.932991,-0.709404
2,March,2016,BARMM,107,3,2016-03-01,0,-1.200942,-0.598167,-1.113502,-0.714004,-0.829756,-0.782601,-0.655927,-0.860060,-0.693572
3,April,2016,BARMM,109,4,2016-04-01,0,-0.841491,-0.550247,-1.174212,-0.643730,-0.701397,-0.761397,-0.627387,-0.880332,-0.701211
4,May,2016,BARMM,165,4,2016-05-01,0,0.130361,0.105918,-0.812353,-0.145349,-0.096060,-0.519818,-0.132426,-0.617627,-0.629541


Phase 2 Complete! Engineered dataset shape: (1020, 16)
